# BERT — Complete Summary

# What BERT Actually Is

BERT is not a new architecture.

It is a Transformer trained in a specific way, on a specific task, with a specific goal.

The goal was simple:

> Train one model so deeply on raw language that anyone can reuse it for almost any NLP task without starting from scratch.

Two words capture BERT completely:

> Pre-train once. Fine-tune anywhere.

---

# How BERT Is Different From A Raw Transformer

A raw Transformer is just the engine.

It is an architecture — attention mechanism, Q K V, contextual vectors.

It does nothing on its own until you train it on something.

BERT is that engine — trained in a specific way by Google on massive text — and then saved and released publicly.

One more important difference:

The original Transformer reads text left to right or right to left.

BERT reads in both directions simultaneously.

When BERT processes the word `"bank"` in:

> `"I went to the bank to deposit money"`

it looks left AND right at the same time.

This is why BERT stands for:

> **Bidirectional Encoder Representations from Transformers**

**Bidirectional** is the key word.

It sees full context from both sides before producing any vector.

---

# How Training Happens

BERT was trained on Wikipedia and a large book corpus.

No labels. Just raw text.

Two training tasks:

---

## Task 1 — Masked Language Modeling (MLM)

Take a sentence.

Randomly mask 15% of words.

Ask BERT to predict the masked words.

Example:

> `"The cat sat on the [MASK]"`

BERT predicts → `"mat"`

Wrong? Nudge Q K V weights.  
Right? Reinforce.

This forces BERT to understand context deeply — because to predict a masked word, you must understand everything around it.

---

## Task 2 — Next Sentence Prediction (NSP)

Show BERT two sentences.

Ask:

> Does sentence B naturally follow sentence A?

### Example 1

Sentence A:

> `"I ordered a pizza"`

Sentence B:

> `"It arrived in 30 minutes"`

Answer → Yes, natural ✅

### Example 2

Sentence A:

> `"I ordered a pizza"`

Sentence B:

> `"Elephants live in Africa"`

Answer → No, random ❌

This forces BERT to understand relationships between sentences, not just within them.

Both tasks run together.

Millions of examples.

Weights get nudged each time.

Eventually BERT develops deep language understanding as a byproduct — exactly like Word2Vec developed word similarity as a byproduct of prediction.

---

# What BERT Produces

Feed any sentence into BERT — it outputs:

- One 768-dimensional vector per token
  - contextual
  - depends on the full sentence

- One special CLS vector
  - summary of the entire sentence
  - used for whole-sentence tasks

Same word in different sentences gets completely different vectors — because context changes the meaning.

---

# WordPiece Tokenization

Before any text reaches BERT — it passes through the WordPiece tokenizer.

Words are broken into known subword pieces.

Example:

> `"unbelievableness"` → `"un" + "believ" + "able" + "ness"`

This happens during training AND inference.

Because of this, BERT almost never encounters a truly unknown word.

It always has pieces to work with.

---

# Fine-Tuning

Pre-trained BERT understands language deeply but does not know your specific task.

You take the saved BERT weights.

Attach a small task-specific layer on top.

Train on your small labeled dataset for a few hours.

> BERT's language knowledge + your task-specific data = powerful model, fast.

Examples:

- Sentiment analysis → attach classification layer
- Question answering → attach span prediction layer
- Named entity recognition → attach label layer per token

---

# Limitations of BERT

## 1. Expensive to run

768 dimensions per token.

Attention across all tokens.

Even inference is slow and memory-heavy compared to simpler models.

---

## 2. Input length limit

BERT can only handle 512 tokens at a time.

Long documents — legal contracts, research papers, books — cannot fit.

You have to chunk them, which creates its own problems.

---

## 3. CLS vector is not a great sentence embedding

This is the big one we will explore next.

The CLS vector exists — but it was trained for classification tasks, not for measuring sentence similarity.

For many real-world use cases, it performs surprisingly poorly.

---

## 4. Masked token limitation

During MLM, only 15% of tokens are masked per sentence.

This means BERT sees each word in its full form most of the time during training.

Some argue this makes training less efficient compared to newer approaches.

---

## 5. NSP is weak

Later research showed NSP did not help as much as originally thought.

Models trained without NSP — like RoBERTa — actually performed better.

So one of BERT's two training tasks turned out to be less useful than expected.

---

## 6. Black box

Like all deep learning — you cannot easily explain why BERT produced a particular vector.

Hard to debug in production.

---

# One-Line Mental Model

```text
Word2Vec     → static vectors, no context, fast, lightweight

Transformer  → the engine, attention, contextual vectors

BERT         → transformer pre-trained bidirectionally
               with MLM + NSP, fine-tunable, 768 dims,
               powerful but expensive, 512 token limit,
               CLS vector has a hidden weakness
```

That hidden weakness in CLS — that is exactly where Sentence Transformers come in.

---

# Understanding 512 Tokens vs 768 Dimensions

## Step 1 — What Is A Token First

Before anything else — let us be concrete about what a token actually is.

A token is just a piece of text after WordPiece tokenization.

Example:

Sentence:

> `"I love eating"`

Tokens:

> `"I"`, `"love"`, `"eat"`, `"##ing"`

So 3 words became 4 tokens.

Token count is what BERT sees.

Not word count.

The 512 token limit means:

> Maximum 512 such pieces.

---

## Step 2 — How Does A Token Become A Vector

When you feed tokens into BERT — the first thing that happens is each token gets converted into a vector.

This conversion happens through something called an embedding lookup table.

Think of it as a giant dictionary:

```text
"I"      → [0.2, 0.8, 0.1, ... ]   768 numbers
"love"   → [0.5, 0.1, 0.9, ... ]   768 numbers
"eat"    → [0.3, 0.7, 0.2, ... ]   768 numbers
"##ing"  → [0.1, 0.4, 0.6, ... ]   768 numbers
```

Every token in BERT's vocabulary has one row in this table.

768 numbers per row.

These starting vectors are called **input embeddings**.

They are NOT contextual yet.

They are just the starting point — like Word2Vec vectors.

Same token = same starting vector regardless of sentence.

---

## Step 3 — Where Do The 768 Dimensions Come From

This is a design choice made by the people who built BERT.

They decided:

> Each token will be represented as a vector of 768 numbers.

Why 768?

Because it gives the model enough space to encode rich, complex meaning.

More dimensions = more capacity to capture nuance.

But also more computation.

Think of it like this:

- Describing a person with 3 numbers → very limited
- Describing a person with 768 numbers → you can capture personality, habits, preferences, relationships, mood, etc.

768 was chosen as a good balance between richness and computational cost.

This is a hyperparameter.

The people who built BERT chose it.

There is also:

- smaller BERT → 512 dimensions
- larger BERT → 1024 dimensions

---

## Step 4 — What Happens Inside BERT To Those Vectors

Now you have 4 tokens.

Each is a 768-dimensional vector.

You feed all 4 into BERT together.

Inside BERT there are 12 layers of Transformer blocks.

At each layer:

- every token looks at every other token through attention
- Q K V happens
- each token borrows meaning from others in proportion to attention scores

After each layer — the vectors get updated.

They become richer.

More contextual.

After all 12 layers — each token has a final 768-dimensional vector that is now deeply contextual.

```text
Input tokens:      "I"    "love"   "eat"   "##ing"
                    ↓       ↓        ↓        ↓

Starting vectors: [768]  [768]   [768]    [768]
                   (not contextual yet)

Layer 1 attention:
every token looks at every other token
vectors get updated

Layer 2 attention:
again

...

Layer 12 attention:
again
                    ↓       ↓        ↓        ↓

Final vectors:    [768]  [768]   [768]    [768]
                  (fully contextual now)
```

The shape does not change.

Still 768 per token.

But the meaning inside those 768 numbers transforms completely as it passes through each layer.

---

## Step 5 — The Exact Relationship Between 512 And 768

Think of the full BERT output as a table:

```text
             Dim1  Dim2  Dim3  ... Dim768

Token 1  →  [0.2,  0.8,  0.1, ..., 0.4]
Token 2  →  [0.5,  0.1,  0.9, ..., 0.2]
Token 3  →  [0.3,  0.7,  0.2, ..., 0.8]
...
Token 512→  [0.1,  0.4,  0.6, ..., 0.3]
```

- 512 rows = maximum number of tokens
- 768 columns = richness of representation per token

The 512 limit exists because attention must happen between every pair of tokens.

So computation grows roughly like:

```text
512 × 512
```

attention score calculations.

The 768 dimensions describe how detailed each token representation is.

These are two completely independent design decisions.

---

# One Clean Analogy

Imagine a meeting room.

- 512 = maximum number of people allowed in the room
- 768 = amount of information on each person's profile card

Everyone in the room talks to everyone else.

Too many people → communication becomes expensive.

Meanwhile each person's profile card can still contain rich detail.

The room size limit and the profile card richness are separate concepts.